In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
# from tsfm_lens.utils import (
#     apply_custom_style,
#     load_dyst_batch,
#     set_seed,
# )

In [ ]:
# read in pickle file
model_name = "chronos-t5-base"
config_key = "rf4_sl4"

induction_results_dir = "../../outputs/rrt_induction_scores"
with open(os.path.join(induction_results_dir, model_name, config_key, "center_scores.pkl"), "rb") as f:
    center_scores = pickle.load(f)
    all_center_scores = center_scores["all_center_scores"]
    center_scores_mean, center_scores_std = center_scores["mean"], center_scores["std"]

with open(os.path.join(induction_results_dir, model_name, config_key, "right_scores.pkl"), "rb") as f:
    right_scores = pickle.load(f)
    all_right_scores = right_scores["all_right_scores"]
    right_scores_mean, right_scores_std = right_scores["mean"], right_scores["std"]

with open(
    os.path.join(induction_results_dir, model_name, config_key, "correct_token_attribution.pkl"),
    "rb",
) as f:
    correct_token_attribution = pickle.load(f)
    all_correct_token_attributions = correct_token_attribution["correct_token_attributions"]
    all_token_attributions = correct_token_attribution["all_token_attributions"]
    correct_token_attribution_mean, correct_token_attribution_std = (
        correct_token_attribution["mean"],
        correct_token_attribution["std"],
    )

with open(os.path.join(induction_results_dir, model_name, config_key, "rrt_vars.pkl"), "rb") as f:
    rrt_vars = pickle.load(f)
    std_factor, overall_scores_mean, overall_scores_std = (
        rrt_vars["std_factor"],
        rrt_vars["overall_scores"][0],
        rrt_vars["overall_scores"][1],
    )

entropy_results_dir = "../../outputs/head_entropy"
with open(os.path.join(entropy_results_dir, "head_entropy_ca.pkl"), "rb") as f:
    head_entropy = pickle.load(f)
    avg_ent_l_h = torch.tensor(head_entropy["avg_ent_l_h"])

In [ ]:
print("=" * 80)
print("DATA SHAPES")
print("=" * 80)
print(f"all_center_scores shape: {all_center_scores.shape}")
print(f"  Description: Center position induction scores")
print(f"  Dimensions: (batch, layers, heads)\n")

print(f"all_right_scores shape: {all_right_scores.shape}")
print(f"  Description: Right position induction scores")
print(f"  Dimensions: (batch, layers, heads)\n")

print(f"all_correct_token_attribution shape: {all_correct_token_attributions.shape}")
print(f"  Description: Attribution scores for correct tokens")
print(f"  Dimensions: (batch, layers, heads)\n")

print(f"all_token_attribution shape: {all_token_attributions.shape}")
print(f"  Description: Attribution scores for all tokens")
print(f"  Dimensions: (batch, layers, heads, tokens)\n")

print(f"avg_ent_l_h shape: {avg_ent_l_h.shape}")
print(f"  Description: Average head entropy")
print(f"  Dimensions: (layers, heads)\n")

# print("=" * 80)
# print("SUMMARY STATISTICS")
# print("=" * 80)
# print(f"Model: {model_name}")
# print(f"Config: {config_key}")
# print(f"Std Factor: {std_factor}\n")

# print(f"Overall Scores:")
# print(f"  Mean: {overall_scores_mean:.4f}")
# print(f"  Std:  {overall_scores_std:.4f}\n")

# print(f"Center Scores: {center_scores_mean.shape} (layers x heads)")
# print(f"  Mean range: [{center_scores_mean.min():.4f}, {center_scores_mean.max():.4f}]")
# print(f"  Std range:  [{center_scores_std.min():.4f}, {center_scores_std.max():.4f}]\n")

# print(f"Right Scores: {right_scores_mean.shape} (layers x heads)")
# print(f"  Mean range: [{right_scores_mean.min():.4f}, {right_scores_mean.max():.4f}]")
# print(f"  Std range:  [{right_scores_std.min():.4f}, {right_scores_std.max():.4f}]\n")

# print(f"Correct Token Attribution: {correct_token_attribution_mean.shape} (layers x heads)")
# print(f"  Mean range: [{correct_token_attribution_mean.min():.4f}, {correct_token_attribution_mean.max():.4f}]")
# print(f"  Std range:  [{correct_token_attribution_std.min():.4f}, {correct_token_attribution_std.max():.4f}]")

In [ ]:
center_scores_mean = all_center_scores.mean(dim=0)
right_scores_mean = all_right_scores.mean(dim=0)
correct_token_attribution_mean = all_correct_token_attributions.mean(dim=0)

In [ ]:
relative_attribution_scores = (all_correct_token_attributions - all_token_attributions.mean(dim=-1))/all_token_attributions.std(dim=-1)
relative_attributions_mean = relative_attribution_scores.mean(dim=0)
relative_attributions_mean.shape

In [ ]:
# 3x3 lower-triangular grid for:
#   (1) right_scores_mean
#   (2) correct_token_attribution_mean
#   (3) avg_ent_l_h

def _to_1d_numpy(a):
    if isinstance(a, torch.Tensor):
        return a.detach().cpu().flatten().numpy()
    return np.asarray(a).flatten()

x_right = _to_1d_numpy(right_scores_mean)
y_attr = _to_1d_numpy(correct_token_attribution_mean)
z_ent = _to_1d_numpy(avg_ent_l_h)  # [12, 12] -> 144

vars_ = [x_right, y_attr, z_ent]
labels = ["Right Scores (Mean)", "Correct Token Attribution (Mean)", "Avg Head Entropy"]

fig, axes = plt.subplots(3, 3, figsize=(12, 12), sharex="col", sharey=False)

for i in range(3):
    for j in range(3):
        ax = axes[i, j]

        if i == j:
            ax.hist(vars_[i], bins=30, alpha=0.8)
            ax.set_title(labels[i])
            ax.set_ylabel("Count")
            ax.grid(True, alpha=0.3)

        elif i > j:
            ax.scatter(vars_[j], vars_[i], alpha=0.6, s=18)
            ax.set_title(f"{labels[j]} vs {labels[i]}")
            ax.grid(True, alpha=0.3)

        else:
            ax.axis("off")

        if i == 2 and i > j:
            ax.set_xlabel(labels[j])
        if j == 0 and i > j:
            ax.set_ylabel(labels[i])

plt.tight_layout()
plt.show()

In [ ]:
correct_token_threshold = 4.0
right_score_threshold = 0.3
entropy_threshold = 4.0

# proportion of correct_token_attribution_mean > 2.0
num_heads = correct_token_attribution_mean.numel()
num_heads_above_threshold = (correct_token_attribution_mean > correct_token_threshold).sum().item()
print("P(correct_token_attribution_mean > T_c):\n", num_heads_above_threshold / num_heads)

# proportion of right_scores_mean > 0.1
num_heads_right_above_threshold = (right_scores_mean > right_score_threshold).sum().item()
print("P(right_scores_mean > T_r):\n", num_heads_right_above_threshold / num_heads)

# proportion of avg_ent_l_h < 4.0
num_heads_entropy_below_threshold = (avg_ent_l_h < entropy_threshold).sum().item()
print("P(avg_ent_l_h < T_e):\n", num_heads_entropy_below_threshold / num_heads)

# proportion of correct_token_attribution_mean > 2.0 and right_scores_mean > 0.1
num_heads_both_above_threshold = ((correct_token_attribution_mean > correct_token_threshold) & (right_scores_mean > right_score_threshold)).sum().item()
print("P(correct_token_attribution_mean > T_c, right_scores_mean > T_r):\n", num_heads_both_above_threshold / num_heads)

print("=" * 40)

# proportion of corrct_token_attribution_mean > 2.0 given right_scores_mean > 0.1
num_heads_right_above_threshold = ((correct_token_attribution_mean > correct_token_threshold) & (right_scores_mean > right_score_threshold)).sum().item()
num_heads_right_condition = (right_scores_mean > right_score_threshold).sum().item()
print("P(correct_token_attribution_mean > T_c | right_scores_mean > T_r):\n", num_heads_right_above_threshold / num_heads_right_condition)

# proportion of avg_ent_l_h < 4.0 given correct_token_attribution_mean > 2.0 and right_scores_mean > 0.1
num_heads_entropy_below_threshold = ((avg_ent_l_h < entropy_threshold) & (correct_token_attribution_mean > correct_token_threshold) & (right_scores_mean > right_score_threshold)).sum().item()
print("P(avg_ent_l_h < T_e | correct_token_attribution_mean > T_c, right_scores_mean > T_r):\n", num_heads_entropy_below_threshold / num_heads_both_above_threshold)

# proportion of correct_token_attribution_mean > 2.0 and right_scores_mean > 0.1 given avg_ent_l_h < 4.0
num_heads_both_above_threshold = ((correct_token_attribution_mean > correct_token_threshold) & (right_scores_mean > right_score_threshold) & (avg_ent_l_h < entropy_threshold)).sum().item()
num_heads_entropy_condition = (avg_ent_l_h < entropy_threshold).sum().item()
print("P(correct_token_attribution_mean > T_c, right_scores_mean > T_r | avg_ent_l_h < T_e):\n", num_heads_both_above_threshold / num_heads_entropy_condition)